## 🌊 Structured Streaming con PySpark y Delta Lake

> **Notebook de cluster Spark** — conecta a un cluster estándar antes de ejecutar.
> Para la demo de Materialized Views, usa el notebook `streaming_II_materialized_views.ipynb` conectado a un SQL Warehouse.

## Escenario

Simulamos una red de sensores industriales con dos tipos de datos:
- **Lecturas de temperatura** (`id`, `id_machine`, `temperature`, `timestamp`)
- **Eventos de estado de máquina** (`id`, `id_machine`, `status`, `timestamp`)

El campo `id_machine` es la clave que permite cruzar ambas fuentes en Silver.

## Estructura del notebook

| Parte | Contenido |
|---|---|
| **Parte 1** | Setup: imports, paths, schemas |
| **Parte 2** | Generación de datos — temperatura (para Partes 3 y 4) |
| **Parte 3** | Structured Streaming con `.json()` |
| **Parte 4** | Structured Streaming con Auto Loader |
| **Parte 5** | Generación de datos para demo MV + ingesta Bronze |
| → | Continúa en `streaming_II_materialized_views.ipynb` |

# PARTE 1 — Setup
---

In [0]:
# ── CELL 1: Imports ─────────────────────────────────────────────
import json
import uuid
import random
import time
import re
from datetime import datetime
from pyspark.sql.types import StructType, StringType, DoubleType, TimestampType
import pyspark.sql.functions as F

## 1.1 Paths y configuración

Toda la información de rutas en **un único sitio**. Cada sección tiene su propio checkpoint y tabla para que los streams sean completamente independientes entre sí.

In [0]:
# ── CELL 2: Paths & config ──────────────────────────────────────
user       = spark.sql("SELECT current_user()").first()[0]
user_clean = re.sub(r"@.*", "", user).replace(".", "_").replace("-", "_")

base_path   = f"s3://mi-bucket-publico-javier-2026/{user_clean}"

# ── Parte 3: .json() ──
source_path      = f"{base_path}/source"                      # JSONs temperatura
checkpoint_json  = f"{base_path}/checkpoints/json_stream"
table_json       = f"{base_path}/tables/json_stream"

# ── Parte 4: Auto Loader ──
checkpoint_al    = f"{base_path}/checkpoints/autoloader_stream"
table_al         = f"{base_path}/tables/autoloader_stream"
schema_location  = f"{base_path}/checkpoints/autoloader_schema"

# ── Parte 5: datos para demo MV ──
source_readings        = f"{base_path}/mv_source/readings"    # JSONs temperatura MV
source_events          = f"{base_path}/mv_source/events"      # JSONs estado máquina
checkpoint_br_readings = f"{base_path}/checkpoints/mv_bronze_readings"
checkpoint_br_events   = f"{base_path}/checkpoints/mv_bronze_events"
schema_loc_readings    = f"{base_path}/checkpoints/mv_schema_readings"
schema_loc_events      = f"{base_path}/checkpoints/mv_schema_events"

# ── Config general ──
schema_name  = "sesion_ss"
num_files    = 10
wait_seconds = 5

print(f"Usuario         : {user_clean}")
print(f"source_path     : {source_path}")
print(f"source_readings : {source_readings}")
print(f"source_events   : {source_events}")

Usuario         : test_data_jm
source_path     : s3://mi-bucket-publico-javier-2026/test_data_jm/source
source_readings : s3://mi-bucket-publico-javier-2026/test_data_jm/mv_source/readings
source_events   : s3://mi-bucket-publico-javier-2026/test_data_jm/mv_source/events


In [0]:
# ── CELL 3: Setup schemas y directorios ─────────────────────────
dbutils.fs.mkdirs(source_path)
dbutils.fs.mkdirs(source_readings)
dbutils.fs.mkdirs(source_events)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")
spark.sql("CREATE SCHEMA IF NOT EXISTS sesion_mv_br")
spark.sql("CREATE SCHEMA IF NOT EXISTS sesion_mv_sv")
print("✅ Schemas y directorios creados.")

✅ Schemas y directorios creados.


# PARTE 2 — Generación de datos: temperatura
---

Generamos datos de temperatura para usar en las Partes 3 y 4 (demo de `.json()` y Auto Loader).

Cada fichero JSON contiene **un único registro** con cuatro campos:
- `id` → identificador único de la lectura
- `id_machine` → máquina que generó la lectura (5 posibles)
- `temperature` → valor entre 18°C y 30°C
- `timestamp` → momento de la lectura

El campo `id_machine` es clave — nos permitirá cruzar las lecturas con el estado de la máquina en la demo de Materialized Views.

In [0]:
# ── CELL 4: Generar JSONs de temperatura ────────────────────────
for i in range(num_files):
    now = datetime.utcnow()
    record = {
        "id":          str(uuid.uuid4()),
        "id_machine":  f"machine_{random.randint(1, 5)}",
        "temperature": round(random.uniform(18.0, 30.0), 2),
        "timestamp":   now.isoformat()
    }
    filepath = f"{source_path}/temperature_{now.strftime('%Y%m%dT%H%M%SZ')}_{i}.json"
    dbutils.fs.put(filepath, json.dumps(record), overwrite=True)
    print(f"✅ [{i+1}/{num_files}] machine={record['id_machine']}  temp={record['temperature']}°C")
    time.sleep(wait_seconds)

Wrote 138 bytes.
✅ [1/10] machine=machine_4  temp=27.59°C
Wrote 138 bytes.
✅ [2/10] machine=machine_5  temp=27.51°C
Wrote 138 bytes.
✅ [3/10] machine=machine_5  temp=22.69°C
Wrote 138 bytes.
✅ [4/10] machine=machine_2  temp=20.87°C
Wrote 138 bytes.
✅ [5/10] machine=machine_5  temp=23.73°C
Wrote 138 bytes.
✅ [6/10] machine=machine_4  temp=18.21°C
Wrote 138 bytes.
✅ [7/10] machine=machine_5  temp=21.06°C
Wrote 138 bytes.
✅ [8/10] machine=machine_2  temp=26.86°C
Wrote 138 bytes.
✅ [9/10] machine=machine_1  temp=26.46°C
Wrote 138 bytes.
✅ [10/10] machine=machine_5  temp=23.83°C


# PARTE 3 — Structured Streaming con `.json()`
---

## ¿Cómo sabe Spark qué ficheros son nuevos?

Cuando usas `.json(source_path)` en un `readStream`, en cada trigger Spark ejecuta:

1. **LIST** del directorio completo en S3
2. **Consulta el checkpoint** → ficheros ya procesados
3. **Anti-join conceptual** → `ficheros_en_S3 − ya_procesados = ficheros_nuevos`
4. **Procesa** solo los ficheros nuevos

Correcto, pero con un problema de escala: **el LIST crece con el número de ficheros acumulados en S3** — con millones de ficheros históricos, cada trigger es costoso aunque solo haya llegado uno nuevo.

## Schema explícito: por qué es obligatorio

En batch, Spark puede inferir el schema escaneando todos los ficheros al inicio. En streaming no puede — el stream no sabe cuántos ficheros llegarán ni cuándo. El schema debe declararse siempre de forma explícita con `.json()` y `.csv()`.

In [0]:
# ── CELL 5: Schema explícito ────────────────────────────────────
# Obligatorio con .json() — Spark no puede inferirlo en streaming
file_schema = StructType() \
    .add("id",          StringType()) \
    .add("id_machine",  StringType()) \
    .add("temperature", DoubleType()) \
    .add("timestamp",   TimestampType())

## 3.1 readStream con `.json()`

### Parámetros en detalle

#### `.schema(file_schema)`
Declara el schema de los ficheros JSON entrantes. Obligatorio si el formato no se autodescribe (`.json()`, `.csv()`). No obligatorio si lleva schema embebido (`.parquet()`, Delta, Auto Loader).

#### `.option("maxFilesPerTrigger", 1)`
Limita cuántos ficheros nuevos se leen por microbatch.

| Valor | Latencia | Throughput | Cuándo |
|---|---|---|---|
| `1` | Alta | Bajo | Demos, desarrollo |
| `10-100` | Media | Medio | Producción moderada |
| Sin opción | Baja | Alto | Backfill rápido |

Nota sobre el overhead: con valor `1` Spark paga el coste fijo de commit Delta y checkpoint update en cada batch. Con valor `100` lo paga una sola vez para 100 ficheros — el coste por fichero es mucho menor.

#### `.json(source_path)`
Define el formato y la ruta. En cada trigger: LIST de S3 + anti-join contra el checkpoint = modelo PULL.

In [0]:
# ── CELL 6: readStream con .json() ──────────────────────────────
raw_stream_df = spark.readStream \
    .schema(file_schema) \
    .option("maxFilesPerTrigger", 1) \
    .json(source_path)

print("isStreaming:", raw_stream_df.isStreaming)

isStreaming: True


## 3.2 Transformación

Las transformaciones sobre un streaming DataFrame son idénticas a batch — mismas funciones, misma API.

Añadimos `processed_timestamp` para ilustrar la diferencia entre:
- **Event time** (`timestamp`): cuándo el sensor generó la lectura
- **Processing time** (`processed_timestamp`): cuándo el stream la procesó

In [0]:
# ── CELL 7: Transformación ──────────────────────────────────────
transformed_stream_df = raw_stream_df \
    .withColumn("processed_timestamp", F.current_timestamp())

## 3.3 writeStream

#### `.format("delta")`
Sink en formato Delta. Transaccional, con schema enforcement y compatible con lectura concurrente en batch.

#### `.outputMode("append")`
Solo filas nuevas de cada microbatch se añaden a la tabla. Único modo compatible con Delta sin agregaciones.

| Modo | Comportamiento | Cuándo |
|---|---|---|
| `append` | Solo filas nuevas del batch | Inserciones simples, sin agregaciones |
| `complete` | Reescribe toda la tabla en cada batch | Agregaciones globales (COUNT total) |
| `update` | Solo filas que han cambiado | Agregaciones con estado (COUNT por ventana) |

#### `.option("checkpointLocation", checkpoint_json)`
Ruta donde Spark persiste el estado del stream. Contiene tres partes:
- `offsets/` → WAL: qué ficheros iba a procesar cada batch (escrito ANTES de ejecutar)
- `commits/` → qué batches se han completado con éxito
- `sources/` → lista acumulada de todos los ficheros ya procesados

La diferencia entre `offsets/` y `commits/` es lo que permite la recuperación exactly-once: si el batch N está en `offsets/` pero no en `commits/`, hay que reprocesarlo.

#### `.trigger(availableNow=True)`
Procesa todos los ficheros pendientes (respetando `maxFilesPerTrigger`) y para cuando termina.

| Trigger | Comportamiento |
|---|---|
| Sin trigger | Microbatches continuos, nunca para |
| `processingTime="30s"` | Un batch cada X tiempo, nunca para |
| `once=True` | Todo en un único batch, para — ignora `maxFilesPerTrigger` |
| `availableNow=True` | Múltiples batches hasta vaciar, para ✅ |

#### `.start(table_json)`
Arranca el stream. Alternativa: `.table("schema.tabla")` para escribir en una tabla registrada en Unity Catalog.

In [0]:
# ── CELL 8: writeStream → S3 (Delta) ────────────────────────────
stream_json = transformed_stream_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_json) \
    .trigger(availableNow=True) \
    .start(table_json)

## 3.4 Dashboard de Structured Streaming

Example
> 📷 <img src="https://raw.githubusercontent.com/jmartinezceste/Course_Delta_Lake/main/delta_img/delta_ses3_2.png" width="1000px"/>

### Gráfica izquierda: Input vs. Processing Rate
- **Input rate** (azul): velocidad a la que llegan datos nuevos a la fuente
- **Processing rate** (naranja): velocidad a la que el stream los consume

| Momento | Qué pasa |
|---|---|
| Arranque | Processing rate (naranja) sube rápido. Input rate (azul) sube más despacio — Spark tarda en descubrir todos los ficheros vía LIST |
| Zona estable | Ambas al mismo nivel — sin lag |
| Final | Processing rate cae primero — se agotan los ficheros pendientes |

### Gráfica derecha: Batch Duration
| Momento | Qué pasa |
|---|---|
| Primer batch | Más lento: JVM warmup, primera conexión a S3, inicialización del stream |
| Batches intermedios | Régimen estable. Con `maxFilesPerTrigger=1` casi todo es overhead fijo (commit Delta + checkpoint update) |
| Último batch | Pico al cierre: el stream detecta que no hay más ficheros y ejecuta el protocolo de terminación |

In [0]:
# ── CELL 9: Esperar a que termine ───────────────────────────────
stream_json.awaitTermination()
print("✅ Stream .json() finalizado.")
print("Status:", stream_json.status)

✅ Stream .json() finalizado.
Status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}


In [0]:
# ── CELL 10: Leer resultado en batch ────────────────────────────
df_json = spark.read.format("delta").load(table_json)
display(df_json)

id,id_machine,temperature,timestamp,processed_timestamp
9f13bca5-3309-4c05-b062-b43c41bba31f,machine_5,20.14,2026-05-06T07:34:43.392Z,2026-05-07T11:40:50.128Z
da7c6bbc-56b9-41a8-8575-c32f62555cf7,machine_5,23.73,2026-05-07T11:39:08.457Z,2026-05-07T11:42:07.622Z
39bd8807-a0ed-4aa0-9056-96bef1a8b07d,machine_2,26.86,2026-05-07T11:39:26.831Z,2026-05-07T11:42:22.113Z
dba09a09-cd6a-4a9d-bf14-29086f9e0ee9,machine_5,22.69,2026-05-07T11:38:56.249Z,2026-05-07T11:41:57.260Z
12c72231-98e8-4ca0-aea9-2e58dda775a4,machine_5,21.06,2026-05-07T11:39:20.880Z,2026-05-07T11:42:17.231Z
50cabcde-dcba-4000-a50c-57c81491c147,machine_5,20.49,2026-05-06T07:35:14.014Z,2026-05-07T11:41:22.840Z
4101cf0a-3668-4fb9-8846-cfedf7c4d9ec,machine_4,20.7,2026-05-06T07:35:25.989Z,2026-05-07T11:41:37.596Z
0712cc22-cee4-46d3-967f-bdfd6f286a07,machine_3,28.11,2026-05-06T07:34:37.506Z,2026-05-06T07:39:22.798Z
aebc4035-80bc-4c41-8125-3ba93e528016,machine_1,24.02,2026-05-06T07:35:07.924Z,2026-05-07T11:41:17.617Z
c0282924-5e40-4784-a6af-2667bd5d8f3e,machine_4,18.21,2026-05-07T11:39:14.439Z,2026-05-07T11:42:12.398Z


# PARTE 4 — Auto Loader
---

## El problema del modelo PULL

Con `.json()`, Spark **pregunta activamente** en cada trigger: hace un LIST completo de S3 y compara contra el checkpoint. Es un modelo **PULL**.

El coste del LIST crece con el número de ficheros acumulados en el bucket. Con millones de ficheros históricos, cada trigger es costoso aunque solo haya llegado un fichero nuevo.

## La solución: modelo PUSH

Auto Loader invierte el flujo. En lugar de que Spark vaya a buscar, **S3 avisa a Spark** cuando llega algo nuevo:

1. Auto Loader configura una cola **SQS** suscrita a eventos S3
2. Cuando llega un fichero nuevo, S3 emite un evento `s3:ObjectCreated`
3. El evento va a la cola SQS
4. Spark consume la cola — sin hacer ningún LIST

**Resultado**: el coste de detección es constante independientemente del volumen histórico acumulado.

## Otras ventajas

- **Schema inference automático**: infiere el schema la primera vez y lo guarda en `schemaLocation`
- **Schema evolution**: si llega un campo nuevo en el JSON, Auto Loader lo detecta y actualiza el schema — con `.json()` ese campo se descartaría silenciosamente
- **Schema no obligatorio**: puedes declararlo si quieres, pero no es necesario

## Diferencia en el código

```python
# .json() — modelo PULL
spark.readStream
    .schema(file_schema)           # obligatorio
    .option("maxFilesPerTrigger", 1)
    .json(source_path)

# Auto Loader — modelo PUSH
spark.readStream
    .format("cloudFiles")          # activa Auto Loader
    .option("cloudFiles.format", "json")       # formato separado
    .option("cloudFiles.schemaLocation", ...)  # donde guardar el schema
    .option("maxFilesPerTrigger", 1)           # igual
    .load(source_path)             # ruta
```

El writeStream no cambia. Auto Loader solo afecta al readStream.

In [0]:
# ── CELL 11: Generar más JSONs para Auto Loader ─────────────────
# Generamos un segundo lote para que Auto Loader tenga datos frescos
for i in range(num_files):
    now = datetime.utcnow()
    record = {
        "id":          str(uuid.uuid4()),
        "id_machine":  f"machine_{random.randint(1, 5)}",
        "temperature": round(random.uniform(18.0, 30.0), 2),
        "timestamp":   now.isoformat()
    }
    filepath = f"{source_path}/temperature_{now.strftime('%Y%m%dT%H%M%SZ')}_{i}.json"
    dbutils.fs.put(filepath, json.dumps(record), overwrite=True)
    print(f"✅ [{i+1}/{num_files}] machine={record['id_machine']}  temp={record['temperature']}°C")
    time.sleep(wait_seconds)

Wrote 138 bytes.
✅ [1/10] machine=machine_3  temp=29.51°C
Wrote 138 bytes.
✅ [2/10] machine=machine_4  temp=25.19°C
Wrote 138 bytes.
✅ [3/10] machine=machine_2  temp=21.74°C
Wrote 138 bytes.
✅ [4/10] machine=machine_4  temp=18.29°C
Wrote 138 bytes.
✅ [5/10] machine=machine_5  temp=26.71°C
Wrote 138 bytes.
✅ [6/10] machine=machine_5  temp=19.65°C
Wrote 138 bytes.
✅ [7/10] machine=machine_4  temp=22.89°C
Wrote 138 bytes.
✅ [8/10] machine=machine_1  temp=25.44°C
Wrote 137 bytes.
✅ [9/10] machine=machine_3  temp=27.1°C
Wrote 138 bytes.
✅ [10/10] machine=machine_1  temp=21.13°C


## 4.1 readStream con Auto Loader

#### `.format("cloudFiles")`
Activa el conector Auto Loader. `cloudFiles` es el identificador que usa Databricks.

#### `.option("cloudFiles.format", "json")`
El formato del fichero fuente. Con `.json()` estaba implícito en el método. Aquí se especifica como opción separada. Admite: `json`, `csv`, `parquet`, `avro`, `text`, `binaryFile`.

#### `.option("cloudFiles.schemaLocation", schema_location)`
Ruta donde Auto Loader persiste el schema inferido. La primera vez infiere el schema leyendo una muestra. Las siguientes lo recupera de aquí sin releer. Si llega un campo nuevo, lo detecta y actualiza automáticamente — con `.json()` ese campo se descartaría silenciosamente.

In [0]:
# ── CELL 12: readStream con Auto Loader ─────────────────────────
raw_stream_al = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.schemaLocation", schema_location) \
    .option("maxFilesPerTrigger", 1) \
    .load(source_path)

print("isStreaming:", raw_stream_al.isStreaming)
print("Schema inferido por Auto Loader:")
raw_stream_al.printSchema()

isStreaming: True
Schema inferido por Auto Loader:
root
 |-- id: string (nullable = true)
 |-- id_machine: string (nullable = true)
 |-- temperature: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [0]:
# ── CELL 13: Transformación ─────────────────────────────────────
transformed_stream_al = raw_stream_al \
    .withColumn("processed_timestamp", F.current_timestamp())

In [0]:
# ── CELL 14: writeStream Auto Loader → Delta ─────────────────────
stream_al = transformed_stream_al.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_al) \
    .trigger(availableNow=True) \
    .start(table_al)

In [0]:
# ── CELL 15: Esperar a que termine ──────────────────────────────
stream_al.awaitTermination()
print("✅ Stream Auto Loader finalizado.")
print("Status:", stream_al.status)

✅ Stream Auto Loader finalizado.
Status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}


In [0]:
# ── CELL 16: Leer resultado en batch ────────────────────────────
df_al = spark.read.format("delta").load(table_al)
display(df_al)

id,id_machine,temperature,timestamp,_rescued_data,processed_timestamp
ee31a59f-3ad1-4011-9708-fa124b144c3f,machine_1,29.92,2026-05-06T07:34:31.401717,null,2026-05-07T11:44:44.100Z
0712cc22-cee4-46d3-967f-bdfd6f286a07,machine_3,28.11,2026-05-06T07:34:37.506399,null,2026-05-07T11:44:44.100Z
9f13bca5-3309-4c05-b062-b43c41bba31f,machine_5,20.14,2026-05-06T07:34:43.392036,null,2026-05-07T11:44:44.100Z
2a838d26-de7c-4dde-8878-dab75c4f2338,machine_4,28.13,2026-05-06T07:34:50.138352,null,2026-05-07T11:44:44.100Z
e5dc50d4-5292-48f5-854c-642899414360,machine_5,26.69,2026-05-06T07:34:56.033830,null,2026-05-07T11:44:44.100Z
353b5d71-050f-4d6c-a947-4a6e52fc1945,machine_4,21.84,2026-05-06T07:35:02.071057,null,2026-05-07T11:44:44.100Z
aebc4035-80bc-4c41-8125-3ba93e528016,machine_1,24.02,2026-05-06T07:35:07.924902,null,2026-05-07T11:44:44.100Z
50cabcde-dcba-4000-a50c-57c81491c147,machine_5,20.49,2026-05-06T07:35:14.014041,null,2026-05-07T11:44:44.100Z
1054697c-28ae-47ea-a3b2-252d1e9b8272,machine_5,18.85,2026-05-06T07:35:19.912305,null,2026-05-07T11:44:44.100Z
5dd8c2a3-94b9-41a9-914f-a8abb7b742f6,machine_4,27.59,2026-05-07T11:38:43.799366,null,2026-05-07T11:44:44.100Z


## Comparativa: `.json()` vs Auto Loader

| | `.json(path)` | Auto Loader (`cloudFiles`) |
|---|---|---|
| Mecanismo de detección | PULL: LIST completo de S3 en cada trigger | PUSH: S3 notifica vía evento SQS |
| Coste con volumen histórico alto | Crece — LIST de millones de ficheros | Constante — solo lee la cola de eventos |
| Schema | Manual y obligatorio | Automático (inferido y persistido) |
| Schema evolution | No — campo nuevo se descarta | Sí — detecta y actualiza automáticamente |
| Configuración | Simple — ideal para demos y aprendizaje | Requiere permisos S3/SQS configurados |
| Motor subyacente | Structured Streaming | Structured Streaming (idéntico) |

# PARTE 5 — Generación de datos para demo de Materialized Views
---

En esta parte preparamos los datos y la capa Bronze para la demo del notebook de Materialized Views.

Generamos **dos tipos de datos** con `id_machine` como clave de cruce:

| Fuente | Campos | Destino |
|---|---|---|
| Lecturas de temperatura | `id`, `id_machine`, `temperature`, `timestamp` | `sesion_mv_br.sensor_readings` |
| Eventos de estado | `id`, `id_machine`, `status`, `timestamp` | `sesion_mv_br.machine_events` |

El campo `status` refleja el estado operacional de la máquina:
- `online` → funcionando con normalidad
- `offline` → detenida
- `maintenance` → en proceso de mantenimiento
- `error` → fallo detectado

En Silver, cruzar temperatura con estado permite responder: *¿hay lecturas anómalas en máquinas que están en error o mantenimiento?*

In [0]:
# ── CELL 17: Generar JSONs de temperatura (MV demo) ─────────────
print("Generando lecturas de temperatura...")
for i in range(num_files):
    now = datetime.utcnow()
    record = {
        "id":          str(uuid.uuid4()),
        "id_machine":  f"machine_{random.randint(1, 5)}",
        "temperature": round(random.uniform(15.0, 35.0), 2),
        "timestamp":   now.isoformat()
    }
    filepath = f"{source_readings}/reading_{now.strftime('%Y%m%dT%H%M%SZ')}_{i}.json"
    dbutils.fs.put(filepath, json.dumps(record), overwrite=True)
    print(f"  🌡️  [{i+1}/{num_files}] machine={record['id_machine']}  temp={record['temperature']}°C")
    time.sleep(2)

Generando lecturas de temperatura...
Wrote 138 bytes.
  🌡️  [1/10] machine=machine_5  temp=29.05°C
Wrote 138 bytes.
  🌡️  [2/10] machine=machine_1  temp=17.92°C
Wrote 138 bytes.
  🌡️  [3/10] machine=machine_1  temp=22.69°C
Wrote 137 bytes.
  🌡️  [4/10] machine=machine_2  temp=24.2°C
Wrote 138 bytes.
  🌡️  [5/10] machine=machine_5  temp=20.55°C
Wrote 138 bytes.
  🌡️  [6/10] machine=machine_5  temp=22.25°C
Wrote 138 bytes.
  🌡️  [7/10] machine=machine_3  temp=32.18°C
Wrote 138 bytes.
  🌡️  [8/10] machine=machine_2  temp=24.47°C
Wrote 138 bytes.
  🌡️  [9/10] machine=machine_2  temp=32.64°C
Wrote 138 bytes.
  🌡️  [10/10] machine=machine_3  temp=27.03°C


In [0]:
# ── CELL 18: Generar JSONs de estado de máquina ─────────────────
statuses = ["online", "online", "online", "maintenance", "error", "offline"]
print("Generando eventos de estado de máquina...")
for machine_id in range(1, 6):
    now = datetime.utcnow()
    record = {
        "id":         str(uuid.uuid4()),
        "id_machine": f"machine_{machine_id}",
        "status":     random.choice(statuses),
        "timestamp":  now.isoformat()
    }
    filepath = f"{source_events}/event_{now.strftime('%Y%m%dT%H%M%SZ')}_{machine_id}.json"
    dbutils.fs.put(filepath, json.dumps(record), overwrite=True)
    print(f"  🔧  machine_{machine_id}  status={record['status']}")
    time.sleep(1)

Generando eventos de estado de máquina...
Wrote 137 bytes.
  🔧  machine_1  status=offline
Wrote 136 bytes.
  🔧  machine_2  status=online
Wrote 135 bytes.
  🔧  machine_3  status=error
Wrote 135 bytes.
  🔧  machine_4  status=error
Wrote 141 bytes.
  🔧  machine_5  status=maintenance


## 5.1 Ingesta Bronze con Auto Loader

Ingestamos ambas fuentes en Bronze usando Auto Loader. El resultado son dos tablas Delta registradas en Unity Catalog bajo el schema `sesion_mv_br`.

Usamos `.table("sesion_mv_br.sensor_readings")` en lugar de `.start(path)` — esto registra la tabla en Unity Catalog, lo que permite que las Materialized Views del notebook SQL puedan referenciarla por nombre.

El writeStream es idéntico al de la Parte 4 — Auto Loader solo cambia el readStream.

In [0]:
# ── CELL 19: Auto Loader → Bronze sensor_readings ───────────────
stream_br_readings = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.schemaLocation", schema_loc_readings) \
    .option("maxFilesPerTrigger", 1) \
    .load(source_readings) \
    .withColumn("ingested_at", F.current_timestamp()) \
    .writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_br_readings) \
    .trigger(availableNow=True) \
    .table("sesion_mv_br.sensor_readings")

stream_br_readings.awaitTermination()
print("✅ Bronze sensor_readings cargado.")
print(f"   Registros: {spark.table('sesion_mv_br.sensor_readings').count()}")

✅ Bronze sensor_readings cargado.
   Registros: 40


In [0]:
# ── CELL 20: Auto Loader → Bronze machine_events ────────────────
stream_br_events = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.schemaLocation", schema_loc_events) \
    .option("maxFilesPerTrigger", 1) \
    .load(source_events) \
    .withColumn("ingested_at", F.current_timestamp()) \
    .writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_br_events) \
    .trigger(availableNow=True) \
    .table("sesion_mv_br.machine_events")

stream_br_events.awaitTermination()
print("✅ Bronze machine_events cargado.")
print(f"   Registros: {spark.table('sesion_mv_br.machine_events').count()}")

✅ Bronze machine_events cargado.
   Registros: 15


In [0]:
# ── CELL 21: Verificar Bronze ────────────────────────────────────
print("=== Bronze: sensor_readings ===")
display(spark.table("sesion_mv_br.sensor_readings"))

print("=== Bronze: machine_events ===")
display(spark.table("sesion_mv_br.machine_events"))

=== Bronze: sensor_readings ===


id,id_machine,temperature,timestamp,_rescued_data,ingested_at
e7d5d6a8-0ba9-4ce2-aeaa-37d7070ffb19,machine_4,34.36,2026-05-06T14:41:02.454063,null,2026-05-06T14:41:59.256Z
db758d1d-5a39-48f0-96ea-082d93e80b1f,machine_4,16.91,2026-05-06T14:41:05.683957,null,2026-05-06T14:41:59.256Z
1f6f7bfd-dc82-4845-9a90-788699bd601b,machine_2,23.12,2026-05-06T14:41:11.483153,null,2026-05-06T14:41:59.256Z
0343b33e-0b9d-4bdf-9e95-cab7c643fdcb,machine_5,23.24,2026-05-06T14:41:14.412475,null,2026-05-06T14:41:59.256Z
b06e3602-228a-4c73-90b4-e7b4d02f4cb7,machine_2,32.57,2026-05-06T14:41:17.503718,null,2026-05-06T14:41:59.256Z
65540457-c87e-4813-b9e3-24d9b81427f3,machine_1,27.05,2026-05-06T14:41:20.383548,null,2026-05-06T14:41:59.256Z
27e5d70d-8cd1-4f4a-9ad7-fe386ea2e39d,machine_5,24.54,2026-05-06T14:41:23.249431,null,2026-05-06T14:41:59.256Z
61ae3149-789d-486b-a968-1fa28370df86,machine_5,16.83,2026-05-06T14:41:26.122298,null,2026-05-06T14:41:59.256Z
444d6507-3f27-4ae0-a83a-46a02180e692,machine_3,22.62,2026-05-06T14:41:30.262994,null,2026-05-06T14:41:59.256Z
f0459f40-2eb5-4e83-923d-145fe08b055c,machine_5,26.3,2026-05-06T14:41:08.576352,null,2026-05-06T14:41:59.256Z


=== Bronze: machine_events ===


id,id_machine,status,timestamp,_rescued_data,ingested_at
da77a677-90a4-4c53-aa9c-3d42ef094b50,machine_5,maintenance,2026-05-07T09:28:54.164608,null,2026-05-07T09:29:53.082Z
6c9cafba-e181-43c1-ac71-eefe14c6a68f,machine_1,offline,2026-05-07T09:28:46.072575,null,2026-05-07T09:29:53.082Z
1d7ca990-5da6-4e40-8840-f50cb5668dbc,machine_2,online,2026-05-07T09:28:48.201104,null,2026-05-07T09:29:53.082Z
2b4f0bff-f38f-4fb2-b6b9-586c3b749a8e,machine_3,online,2026-05-07T09:28:50.279676,null,2026-05-07T09:29:53.082Z
ff780ca1-68df-445e-9ac1-ddcf9b115b44,machine_4,error,2026-05-07T09:28:52.197213,null,2026-05-07T09:29:53.082Z
6abec4b1-7329-43ca-b15a-2843f7af1b69,machine_5,maintenance,2026-05-07T11:45:40.468486,null,2026-05-07T11:46:31.073Z
ec8ecdae-4f30-4dc4-9c51-87b540990935,machine_1,offline,2026-05-07T11:45:32.585611,null,2026-05-07T11:46:31.073Z
ddc12444-0409-4a1e-8aaf-ccb2698e9bc8,machine_2,online,2026-05-07T11:45:34.637731,null,2026-05-07T11:46:31.073Z
be37c7a6-678c-405c-a7ba-f8836febf33c,machine_3,error,2026-05-07T11:45:36.676345,null,2026-05-07T11:46:31.073Z
a0efab18-ba2d-40b3-8636-44a3814af576,machine_4,error,2026-05-07T11:45:38.594419,null,2026-05-07T11:46:31.073Z


In [0]:
# ── CELL 22: Handoff al notebook de Materialized Views ──────────
readings_count = spark.table("sesion_mv_br.sensor_readings").count()
events_count   = spark.table("sesion_mv_br.machine_events").count()

print("=" * 55)
print("✅  Bronze listo para la demo de Materialized Views")
print("=" * 55)
print(f"  sesion_mv_br.sensor_readings : {readings_count} registros")
print(f"  sesion_mv_br.machine_events  : {events_count} registros")
print()
print("👉  Continúa en:")
print("    streaming_II_materialized_views.ipynb")
print("    (conectado a SQL Warehouse)")
print("=" * 55)

✅  Bronze listo para la demo de Materialized Views
  sesion_mv_br.sensor_readings : 40 registros
  sesion_mv_br.machine_events  : 15 registros

👉  Continúa en:
    streaming_II_materialized_views.ipynb
    (conectado a SQL Warehouse)
